In [21]:
# =========================
# BLOCK 3 — COMPARISON: CLR baseline vs PCRR ratios (noRFE)
# =========================
import os
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# -------------------------
# CONFIG
# -------------------------
clr_dir = "/Users/adithyamadduri/Desktop/Projects/ratios_project/Revision/Results/CLR_baseline_LGBM"
ratio_dir = "/Users/adithyamadduri/Desktop/Projects/ratios_project/Revision/Results/Protein_ratios_noRFE"

classes = ["MCI", "NCI", "AD", "AD+"]
seeds = [1, 2, 3, 4, 5]

def safe_cls(c):
    return c.replace("+", "plus").replace(" ", "_").replace("/", "-")

def compute_auc_from_csv(csv_path):
    df = pd.read_csv(csv_path)
    y_true = df["y_true"].values
    y_score = df["y_score"].values

    # avoid crash if only one class appears in test split
    if np.sum(y_true) == 0 or np.sum(y_true) == len(y_true):
        return np.nan

    return roc_auc_score(y_true, y_score)

# -------------------------
# MAIN: compute mean AUC per class
# -------------------------
rows = []

for cls in classes:
    clr_aucs = []
    ratio_aucs = []

    for seed in seeds:
        clr_csv = os.path.join(clr_dir, f"seed{seed}_{safe_cls(cls)}.csv")
        ratio_csv = os.path.join(ratio_dir, f"seed{seed}_{safe_cls(cls)}.csv")

        if os.path.exists(clr_csv):
            clr_aucs.append(compute_auc_from_csv(clr_csv))
        else:
            clr_aucs.append(np.nan)

        if os.path.exists(ratio_csv):
            ratio_aucs.append(compute_auc_from_csv(ratio_csv))
        else:
            ratio_aucs.append(np.nan)

    clr_mean = np.nanmean(clr_aucs)
    ratio_mean = np.nanmean(ratio_aucs)
    diff = ratio_mean - clr_mean

    rows.append({
        "class": cls,
        "CLR_LGBM": clr_mean,
        "PCRR_ratios_LGBM": ratio_mean,
        "difference (PCRR - CLR)": diff,
        "n_clr": np.sum(~np.isnan(clr_aucs)),
        "n_ratios": np.sum(~np.isnan(ratio_aucs)),
    })

# -------------------------
# OUTPUT TABLE
# -------------------------
out_df = pd.DataFrame(rows).set_index("class")

# keep only the 3 key columns
final_df = out_df[["CLR_LGBM", "PCRR_ratios_LGBM", "difference (PCRR - CLR)"]].copy()

print("\n=== Mean ROC AUC (5 seeds): CLR vs PCRR ===")
print(final_df.round(4))

# optional: show if any seeds were missing
if (out_df["n_clr"] < len(seeds)).any() or (out_df["n_ratios"] < len(seeds)).any():
    print("\nNOTE: Some CSVs were missing (or had invalid AUC due to single-class test split).")
    print(out_df[["n_clr", "n_ratios"]])


=== Mean ROC AUC (5 seeds): CLR vs PCRR ===
       CLR_LGBM  PCRR_ratios_LGBM  difference (PCRR - CLR)
class                                                     
MCI      0.6371            0.6457                   0.0087
NCI      0.7892            0.7469                  -0.0422
AD       0.8314            0.8407                   0.0093
AD+      0.8534            0.8136                  -0.0398


In [22]:
# =========================
# BLOCK 3 — COMPARISON: Z-scale baseline vs PCRR ratios (noRFE)
# =========================
import os
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# -------------------------
# CONFIG
# -------------------------
zscale_dir = "/Users/adithyamadduri/Desktop/Projects/ratios_project/Revision/Results/Zscale_baseline_LGBM"
ratio_dir = "/Users/adithyamadduri/Desktop/Projects/ratios_project/Revision/Results/Protein_ratios_noRFE"

classes = ["MCI", "NCI", "AD", "AD+"]
seeds = [1, 2, 3, 4, 5]

def safe_cls(c):
    return c.replace("+", "plus").replace(" ", "_").replace("/", "-")

def compute_auc_from_csv(csv_path):
    df = pd.read_csv(csv_path)
    y_true = df["y_true"].values
    y_score = df["y_score"].values

    # avoid crash if only one class appears in test split
    if np.sum(y_true) == 0 or np.sum(y_true) == len(y_true):
        return np.nan

    return roc_auc_score(y_true, y_score)

# -------------------------
# MAIN: compute mean AUC per class
# -------------------------
rows = []

for cls in classes:
    zscale_aucs = []
    ratio_aucs = []

    for seed in seeds:
        zscale_csv = os.path.join(zscale_dir, f"seed{seed}_{safe_cls(cls)}.csv")
        ratio_csv = os.path.join(ratio_dir, f"seed{seed}_{safe_cls(cls)}.csv")

        if os.path.exists(zscale_csv):
            zscale_aucs.append(compute_auc_from_csv(zscale_csv))
        else:
            zscale_aucs.append(np.nan)

        if os.path.exists(ratio_csv):
            ratio_aucs.append(compute_auc_from_csv(ratio_csv))
        else:
            ratio_aucs.append(np.nan)

    zscale_mean = np.nanmean(zscale_aucs)
    ratio_mean = np.nanmean(ratio_aucs)
    diff = ratio_mean - zscale_mean

    rows.append({
        "class": cls,
        "Zscale_LGBM": zscale_mean,
        "PCRR_ratios_LGBM": ratio_mean,
        "difference (PCRR - Zscale)": diff,
        "n_zscale": np.sum(~np.isnan(zscale_aucs)),
        "n_ratios": np.sum(~np.isnan(ratio_aucs)),
    })

# -------------------------
# OUTPUT TABLE
# -------------------------
out_df = pd.DataFrame(rows).set_index("class")

# keep only the 3 key columns
final_df = out_df[["Zscale_LGBM", "PCRR_ratios_LGBM", "difference (PCRR - Zscale)"]].copy()

print("\n=== Mean ROC AUC (5 seeds): Z-scale vs PCRR ===")
print(final_df.round(4))

# optional: show if any seeds were missing
if (out_df["n_zscale"] < len(seeds)).any() or (out_df["n_ratios"] < len(seeds)).any():
    print("\nNOTE: Some CSVs were missing (or had invalid AUC due to single-class test split).")
    print(out_df[["n_zscale", "n_ratios"]])


=== Mean ROC AUC (5 seeds): Z-scale vs PCRR ===
       Zscale_LGBM  PCRR_ratios_LGBM  difference (PCRR - Zscale)
class                                                           
MCI         0.6886            0.6457                     -0.0429
NCI         0.7594            0.7469                     -0.0125
AD          0.8473            0.8407                     -0.0066
AD+         0.8268            0.8136                     -0.0132
